# MGMT298D: Science and Strategy of AI
## Assignment 3 - Clustering and Collaborative Filtering
### Application: Netflix Recommendations

---

**Instructions:** Complete the exercises by filling in the `???` placeholders and answering the questions. Run all code cells in order.

## Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
import seaborn as sns

df = pd.read_csv("https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/refs/heads/main/netflix_ratings.csv")
print(f"Loaded {len(df):,} users, {df.shape[1] - 1} movies")

## Data Preparation

In [ ]:
# Fill missing ratings with movie averages, standardize for clustering
data_filled = df.fillna(df.mean(numeric_only=True))
X = data_filled.drop("user_id", axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Features: {X.shape[1]} movies")

## Part 1: Choosing k for Clustering

In [ ]:
# Test different values of k
k_range = range(2, 11)
results = {"k": [], "inertia": [], "silhouette": []}

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_scaled)
    results["k"].append(k)
    results["inertia"].append(kmeans.inertia_)
    results["silhouette"].append(silhouette_score(X_scaled, labels))

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# Visualize elbow and silhouette
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(results_df["k"], results_df["inertia"], marker="o", color="#e63946", linewidth=2)
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")
axes[0].grid(alpha=0.3)

axes[1].plot(results_df["k"], results_df["silhouette"], marker="o", color="#1d3557", linewidth=2)
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Analysis")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Part 2: Clustering with Different k Values

In [ ]:
# Compare k=3 vs k=5
for k in [3, 5]:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto")
    labels = kmeans.fit_predict(X_scaled)
    
    print(f"\n=== k = {k} ===")
    print(f"Cluster sizes: {np.bincount(labels)}")
    print(f"Silhouette: {silhouette_score(X_scaled, labels):.3f}")

In [ ]:
# Final clustering with chosen k
chosen_k = ???  # Fill in: 3, 4, or 5

kmeans = KMeans(n_clusters=chosen_k, random_state=42, n_init="auto")
data_filled["cluster"] = kmeans.fit_predict(X_scaled)

print(f"Cluster distribution with k={chosen_k}:")
print(data_filled["cluster"].value_counts().sort_index())

## Part 3: Analyzing Cluster Profiles

In [ ]:
# Calculate average ratings per cluster
cluster_means = data_filled.groupby("cluster")[X.columns].mean()

# Top 5 movies for each cluster
for cluster in range(chosen_k):
    top5 = cluster_means.loc[cluster].nlargest(5)
    print(f"\n--- Cluster {cluster} ({(data_filled['cluster']==cluster).sum()} users) ---")
    for movie, rating in top5.items():
        print(f"  {movie}: {rating:.2f}")

In [ ]:
# Heatmap of cluster preferences (top 12 most variable movies)
top_movies = X.var().nlargest(12).index

plt.figure(figsize=(14, 5))
sns.heatmap(cluster_means[top_movies], annot=True, cmap="YlOrRd", fmt=".2f")
plt.xlabel("Movies")
plt.ylabel("Cluster")
plt.title("Average Ratings by Cluster (Top 12 Variable Movies)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Part 4: Collaborative Filtering

In [ ]:
# Prepare data for collaborative filtering
user_item = data_filled.drop(["user_id", "cluster"], axis=1)

# Convert to long format for train/test split
ratings_long = user_item.stack().reset_index()
ratings_long.columns = ["user_idx", "movie", "rating"]

train, test = train_test_split(ratings_long, test_size=0.2, random_state=42)
print(f"Train: {len(train):,} ratings | Test: {len(test):,} ratings")

In [ ]:
# Build training matrix and compute similarity
train_matrix = train.pivot(index="user_idx", columns="movie", values="rating")
train_matrix = train_matrix.reindex(index=user_item.index, columns=user_item.columns)
train_matrix = train_matrix.fillna(user_item.mean())

user_sim = pd.DataFrame(
    cosine_similarity(train_matrix),
    index=train_matrix.index,
    columns=train_matrix.index
)

# Prediction function
def predict(user_idx, movie, k=10):
    sims = user_sim.loc[user_idx].drop(user_idx).nlargest(k)
    neighbor_ratings = train_matrix.loc[sims.index, movie]
    return np.dot(sims.values, neighbor_ratings.values) / sims.sum()

In [ ]:
# Evaluate with default k=10
test["pred"] = test.apply(lambda r: predict(r["user_idx"], r["movie"], k=10), axis=1)
mae_default = (test["rating"] - test["pred"]).abs().mean()
print(f"Collaborative Filtering (k=10) — MAE = {mae_default:.3f}")

## Part 5: Tuning the Number of Neighbors

In [ ]:
# Test different values of k
k_values = [5, 10, 20, 50, 100, 200]
mae_results = []

for k in k_values:
    preds = test.apply(lambda r: predict(r["user_idx"], r["movie"], k=k), axis=1)
    mae = (test["rating"] - preds).abs().mean()
    mae_results.append(mae)
    print(f"k={k:3d} — MAE = {mae:.3f}")

best_k = k_values[np.argmin(mae_results)]
best_mae = min(mae_results)
print(f"\nBest: k={best_k} with MAE={best_mae:.3f}")

In [ ]:
# Visualize k vs MAE
plt.figure(figsize=(8, 4))
plt.plot(k_values, mae_results, marker="o", color="#45b7d1", linewidth=2, markersize=8)
plt.axhline(y=best_mae, color="#e63946", linestyle="--", alpha=0.7, label=f"Best MAE = {best_mae:.3f}")
plt.xlabel("Number of Neighbors (k)")
plt.ylabel("MAE")
plt.title("Collaborative Filtering: Effect of k")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot: Actual vs Predicted
test["pred_best"] = test.apply(lambda r: predict(r["user_idx"], r["movie"], k=best_k), axis=1)

sample = test.sample(2000, random_state=42)

plt.figure(figsize=(7, 7))
plt.scatter(sample["rating"], sample["pred_best"], alpha=0.3, color="#1d3557")
plt.plot([1, 5], [1, 5], "r--", linewidth=2, label="Perfect Prediction")
plt.xlabel("Actual Rating")
plt.ylabel("Predicted Rating")
plt.title(f"Actual vs Predicted (k={best_k})")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

---
## Questions

### Question 1: Choosing k for Clustering

**Q1a:** Looking at the elbow plot and silhouette scores, what value of k would you recommend? Explain your reasoning.

*Your answer:*



**Q1b:** The silhouette scores are relatively low (< 0.1). Does this mean clustering is useless for this problem? Why might user preference data be hard to cluster cleanly?

*Your answer:*



---
### Question 2: Business Value of Clusters

**Q2:** Look at the cluster profiles (top movies per cluster). How would Netflix use these segments for:
- (a) Marketing campaigns?
- (b) Content acquisition decisions?

*Your answer:*



---
### Question 3: Collaborative Filtering Trade-offs

**Q3a:** Why does increasing k (number of neighbors) initially improve MAE but eventually plateau or worsen?

*Your answer:*



**Q3b:** A product manager asks: "Why not just use k=1000 to be safe?" What's wrong with this reasoning?

*Your answer:*



---
### Question 4: Cold Start Problem

**Q4:** A new user signs up and has only rated 2 movies. Why would collaborative filtering struggle to make good recommendations for this user? Suggest one approach Netflix could use to handle new users.

*Your answer:*



---
### Question 5: Clustering vs Collaborative Filtering

**Q5:** When would you recommend clustering-based recommendations over user-based collaborative filtering, and vice versa?

*Your answer:*

